# Regresión Poisson en manufactura
### Predecir el número de defectos por lote

Este notebook presenta una técnica estadística para estimar conteos en un proceso de manufactura. La idea central es sencilla: usar datos históricos de producción para estimar cuántos defectos podrían aparecer en un lote nuevo.

En regresión Poisson queremos predecir un **conteo**:

- cuántos defectos tendrá un lote,
- cuántos rechazos aparecerán en un turno,
- cuántos paros tendrá una máquina,
- o cuántos reclamos surgirán en una semana.

En este ejercicio usaremos un caso ficticio de manufactura:

**predecir cuántos defectos tendrá un lote de producción usando mediciones del proceso.**

El recorrido será gradual: primero entenderemos el problema, después revisaremos los
datos, entrenaremos el modelo, evaluaremos sus errores y finalmente interpretaremos
la función matemática con lenguaje sencillo.

**Nivel:** principiante.  
**Datos:** simulados dentro del notebook.  
**Software:** Python, pandas, matplotlib, seaborn y scikit-learn.  
**Equipo:** Google Colab con CPU; no requiere GPU ni archivos externos.

Para correrlo en Colab:

1. Abra https://colab.research.google.com/.
2. Suba este archivo `.ipynb`.
3. Seleccione **Entorno de ejecución -> Ejecutar todas**.
4. Lea la explicación antes de cada bloque y la interpretación después de cada resultado.

## Contexto del problema

Imagine una planta que produce lotes de piezas metálicas. Cada lote contiene cientos
de piezas. Al final del lote, calidad registra cuántos defectos se encontraron.

Para cada lote tenemos estas variables:

| Variable | Qué representa |
|---|---|
| tamaño_lote | Número de piezas producidas en el lote |
| temperatura_c | Temperatura promedio de la máquina |
| vibración_mm_s | Vibración promedio durante el lote |
| velocidad_piezas_hora | Velocidad promedio de producción |
| turno | Turno de producción: día o noche |

La variable que queremos predecir es:

| Variable objetivo | Significado |
|---|---|
| defectos_lote | Número de defectos encontrados en el lote |

La pregunta del ejercicio es:

**Con variables del proceso, podemos estimar cuántos defectos tendrá un lote?**

Este enfoque puede ayudar a priorizar inspección, detectar lotes con mayor riesgo y
entender qué condiciones se asocian con más defectos. No reemplaza la inspección real.

## Qué es la regresión Poisson

La regresión Poisson es una técnica para predecir **conteos**. Un conteo es un número
entero que representa cuántas veces ocurrió algo.

Ejemplos de conteos:

- 0 defectos en un lote,
- 3 paros en una máquina,
- 12 rechazos en un turno,
- 5 reclamos de cliente en una semana.

La regresión Poisson es útil cuando la respuesta:

1. Es un número entero.
2. No puede ser negativa.
3. Representa eventos que ocurren dentro de una unidad: lote, turno, día, semana o máquina.
4. Suele tener muchos valores pequeños.

La idea sencilla es:

**el modelo calcula el número esperado de eventos.**

En este caso, calcula el número esperado de defectos por lote.

La fórmula general se escribe así:

$$
\log(\text{defectos esperados}) =
b_0 + b_1x_1 + b_2x_2 + b_3x_3 + ...
$$

Usamos logaritmo porque el conteo esperado no debe ser negativo. Después, el modelo
convierte ese resultado de regreso a una cantidad positiva de defectos.

En palabras simples:

**Poisson combina las variables del proceso y estima cuántos defectos espera ver.**

## Cuándo conviene usar regresión Poisson

Conviene explorar Poisson cuando la pregunta principal empieza con:

- ¿Cuántos defectos habrá?
- ¿Cuántos eventos esperamos?
- ¿Cuántas fallas ocurrirán?
- ¿Cuántos rechazos aparecerán?

Casos típicos en manufactura:

- Defectos por lote.
- Rechazos por turno.
- Paros por máquina.
- Reclamos por semana.
- Incidentes de calidad por línea.

No es la mejor primera opción cuando:

- La respuesta es una categoría, como Conforme/Retrabajo/Rechazo. Ahí conviene un
  modelo de clasificación, como árbol de decisión o regresión logística.
- El conteo tiene demasiados ceros y un comportamiento especial.
- La variación real es mucho mayor que la que Poisson espera. A eso se le llama
  sobredispersión.
- Los lotes tienen tamaños muy diferentes y necesitamos modelar tasas con exposición.

En este notebook lo mantendremos simple: lotes de tamaños parecidos y un modelo
Poisson de scikit-learn para explicar la técnica.

## 1. Preparar bibliotecas

Este bloque prepara el entorno de Python.

Carga o instala las bibliotecas necesarias:

- `numpy`: genera datos simulados.
- `pandas`: organiza tablas.
- `matplotlib` y `seaborn`: hacen gráficas.
- `scikit-learn`: contiene `PoissonRegressor`.
- `joblib`: guarda el modelo entrenado.

Si Colab ya tiene las bibliotecas, esta celda solo mostrará un mensaje de confirmación.

In [ ]:
import importlib.util
import subprocess
import sys

paquetes = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

faltantes = [paquete for modulo, paquete in paquetes.items()
             if importlib.util.find_spec(modulo) is None]

if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *faltantes])

print("Bibliotecas listas. Podemos continuar.")

## 2. Importar herramientas y definir parámetros

Este bloque importa las herramientas que se usarán en el resto del notebook.

También define:

- `SEMILLA`: permite repetir los mismos resultados.
- `CARPETA`: lugar donde se guardarán datos, gráficas, modelo y conclusiones.
- `guardar_figura`: función pequeña para guardar cada gráfica sin repetir código.

Todavía no entrenamos ningún modelo. Solo dejamos listo el espacio de trabajo.

In [ ]:
from pathlib import Path
import json
import platform
import shutil

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sklearn
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 42
CARPETA = Path("resultados_poisson_manufactura")
CARPETA.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

def guardar_figura(nombre):
    plt.gcf().savefig(CARPETA / f"{nombre}.png", dpi=160, bbox_inches="tight")
    plt.show()

print("Versión de scikit-learn:", sklearn.__version__)
print("Carpeta de resultados:", CARPETA.resolve())

## 3. Crear datos simulados

Este bloque crea un historial ficticio de 500 lotes.

Cada lote tendrá:

- tamaño del lote,
- temperatura promedio,
- vibración promedio,
- velocidad de producción,
- turno,
- y número de defectos.

Para simular los defectos usamos una idea sencilla:

- más vibración tiende a aumentar defectos,
- mayor temperatura tiende a aumentar defectos,
- lotes más grandes tienden a tener más oportunidades de defecto,
- mayor velocidad tiende a aumentar defectos,
- el turno noche tendrá un pequeño aumento simulado.

Para que el ejercicio sea más claro en clase, la simulación usa una señal de proceso
visible. Esto permite que las métricas mejoren frente a una regla simple y facilita
explicar por qué el modelo aprende. Aun así, se conserva variación aleatoria: el modelo
no será perfecto.

Después usamos `rng.poisson(media_defectos)` para crear conteos enteros. Esa línea es
la que hace que la variable objetivo sea compatible con un ejemplo de Poisson.

In [ ]:
rng = np.random.default_rng(SEMILLA)
n_lotes = 500

tamaño_lote = rng.integers(450, 701, size=n_lotes)
temperatura_c = rng.normal(66, 4.0, size=n_lotes)
vibracion_mm_s = rng.normal(2.4, 0.55, size=n_lotes)
velocidad_piezas_hora = rng.normal(120, 12, size=n_lotes)
turno = rng.choice(["Día", "Noche"], size=n_lotes, p=[0.62, 0.38])

efecto_turno_noche = (turno == "Noche") * 0.18

log_media_defectos = (
    -1.20
    + 0.0048 * tamaño_lote
    + 0.060 * (temperatura_c - 66)
    + 0.750 * (vibracion_mm_s - 2.4)
    + 0.010 * (velocidad_piezas_hora - 120)
    + efecto_turno_noche
)

media_defectos = np.exp(log_media_defectos)
defectos_lote = rng.poisson(media_defectos)

datos = pd.DataFrame({
    "id_lote": [f"L{i:04d}" for i in range(1, n_lotes + 1)],
    "tamaño_lote": tamaño_lote,
    "temperatura_c": temperatura_c,
    "vibracion_mm_s": vibracion_mm_s,
    "velocidad_piezas_hora": velocidad_piezas_hora,
    "turno": turno,
    "defectos_lote": defectos_lote,
})

display(datos.head(10).round(2))
print(f"Historial generado: {len(datos)} lotes simulados.")

## 4. Revisar los datos

Antes de entrenar, revisamos si los datos tienen sentido.

Este bloque muestra:

- valores faltantes,
- resumen numérico,
- conteo de lotes por turno,
- y distribución del número de defectos.

La gráfica de defectos es importante porque Poisson se usa para conteos. Debemos ver
números enteros, no negativos, y normalmente muchos valores pequeños.

In [ ]:
print("Valores faltantes por columna:")
display(datos.isna().sum().to_frame("faltantes"))

display(datos.describe(include="all").round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=datos, x="turno", hue="turno", palette={"Día": "#167D9A", "Noche": "#DA9C20"},
              legend=False, ax=axes[0])
axes[0].set_title("Lotes por turno")
axes[0].set_xlabel("Turno")
axes[0].set_ylabel("Lotes")

sns.histplot(data=datos, x="defectos_lote", bins=range(0, datos["defectos_lote"].max() + 2),
             color="#C24E56", ax=axes[1])
axes[1].set_title("Distribución de defectos por lote")
axes[1].set_xlabel("Defectos en el lote")
axes[1].set_ylabel("Lotes")

fig.tight_layout()
guardar_figura("01_revision_datos")

**Interpretación sencilla:** la variable objetivo es un conteo. Hay lotes con pocos
defectos y algunos con más defectos. Ese patrón hace razonable explorar regresión
Poisson como primer modelo.

## 5. Explorar relaciones visualmente

Ahora miramos si las variables del proceso parecen relacionadas con los defectos.

Usaremos diagramas de dispersión:

- Cada punto es un lote.
- El eje horizontal muestra una medición del proceso.
- El eje vertical muestra cuántos defectos tuvo el lote.
- El color indica el turno.

Si los puntos suben conforme aumenta una variable, podría existir una asociación positiva.
Esto no prueba causalidad; solo ayuda a entender los datos antes de modelar.

In [ ]:
variables_numericas = ["tamaño_lote", "temperatura_c", "vibracion_mm_s", "velocidad_piezas_hora"]
etiquetas = ["Tamaño de lote", "Temperatura (C)", "Vibración (mm/s)", "Velocidad (piezas/hora)"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, variable, etiqueta in zip(axes.ravel(), variables_numericas, etiquetas):
    sns.scatterplot(data=datos, x=variable, y="defectos_lote", hue="turno",
                    palette={"Día": "#167D9A", "Noche": "#DA9C20"}, alpha=0.75, ax=ax)
    ax.set_title(f"Defectos vs. {etiqueta}")
    ax.set_xlabel(etiqueta)
    ax.set_ylabel("Defectos")
    ax.legend(title="Turno")

fig.tight_layout()
guardar_figura("02_relaciones_variables")

display(datos.groupby("turno")["defectos_lote"].agg(["count", "mean", "median"]).round(2))

**Interpretación sencilla:** buscamos señales generales, no reglas perfectas. En datos
reales suele haber mucho ruido: dos lotes con condiciones parecidas pueden tener
diferente número de defectos.

## 6. Separar entrenamiento y prueba

Separamos los datos en dos partes para medir si el modelo puede funcionar con lotes
que no ha visto antes.

- Entrenamiento: el modelo aprende.
- Prueba: el modelo se evalúa con lotes que no vio antes.

La variable `X` contiene las entradas. La variable `y` contiene el conteo que queremos
predecir: `defectos_lote`.

In [ ]:
X = datos[["tamaño_lote", "temperatura_c", "vibracion_mm_s", "velocidad_piezas_hora", "turno"]]
y = datos["defectos_lote"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEMILLA
)

print("Lotes para entrenamiento:", len(X_train))
print("Lotes para prueba:", len(X_test))
print("Promedio de defectos en entrenamiento:", round(y_train.mean(), 2))
print("Promedio de defectos en prueba:", round(y_test.mean(), 2))

## 7. Construir y entrenar el modelo Poisson

Este bloque crea el modelo.

Usamos un `Pipeline` con dos partes:

1. **Preparación de variables**
   - Las variables numéricas se estandarizan con `StandardScaler`.
   - La variable `turno` se convierte en columnas numéricas con `OneHotEncoder`.

2. **Modelo**
   - `PoissonRegressor` aprende a predecir el número esperado de defectos.

`fit(X_train, y_train)` entrena el modelo solamente con los lotes de entrenamiento.

In [ ]:
variables_numericas = ["tamaño_lote", "temperatura_c", "vibracion_mm_s", "velocidad_piezas_hora"]
variables_categoricas = ["turno"]

preparacion = ColumnTransformer([
    ("numericas", StandardScaler(), variables_numericas),
    ("categoricas", OneHotEncoder(drop="first"), variables_categoricas),
])

modelo_poisson = Pipeline([
    ("preparacion", preparacion),
    ("poisson", PoissonRegressor(alpha=0.01, max_iter=1000))
])

modelo_poisson.fit(X_train, y_train)
print("Modelo Poisson entrenado correctamente.")

## 8. Evaluar el modelo

Ahora predecimos defectos en los lotes de prueba.

Usaremos varias métricas:

- `MAE`: error absoluto medio. Se interpreta como error promedio en defectos.
- `mean_poisson_deviance`: métrica especial para modelos Poisson. Menor es mejor.
- `RMSE`: error cuadrático medio con raíz. Castiga más los errores grandes.
- `R²`: compara al modelo contra predecir siempre el promedio.

También comparamos contra una regla simple: predecir siempre el promedio de defectos
observado en entrenamiento.

Para principiantes, el MAE suele ser la métrica más fácil de leer porque está en la
unidad del problema: defectos por lote. R² se agrega porque es una métrica conocida,
pero en modelos de conteo debe leerse como una referencia complementaria.

In [ ]:
pred_poisson = modelo_poisson.predict(X_test)

modelo_simple = DummyRegressor(strategy="mean")
modelo_simple.fit(X_train, y_train)
pred_simple = modelo_simple.predict(X_test)

mae_poisson = mean_absolute_error(y_test, pred_poisson)
mae_simple = mean_absolute_error(y_test, pred_simple)
rmse_poisson = np.sqrt(mean_squared_error(y_test, pred_poisson))
rmse_simple = np.sqrt(mean_squared_error(y_test, pred_simple))
r2_poisson = r2_score(y_test, pred_poisson)
r2_simple = r2_score(y_test, pred_simple)
dev_poisson = mean_poisson_deviance(y_test, pred_poisson)
dev_simple = mean_poisson_deviance(y_test, pred_simple)

tabla_metricas = pd.DataFrame({
    "modelo": ["Regla simple: promedio histórico", "Regresión Poisson"],
    "MAE_error_promedio": [mae_simple, mae_poisson],
    "RMSE_penaliza_errores_grandes": [rmse_simple, rmse_poisson],
    "R2": [r2_simple, r2_poisson],
    "desviacion_poisson": [dev_simple, dev_poisson],
})

display(tabla_metricas.round(3))
print(f"Error promedio de Poisson: {mae_poisson:.2f} defectos por lote.")

**Interpretación sencilla:** si el MAE es 1.30, significa que el modelo se equivoca,
en promedio, por 1.30 defectos por lote. No quiere decir que todos los lotes tengan
exactamente ese error; es un promedio.

El RMSE también mide error, pero castiga más los errores grandes. R² compara al modelo
contra una predicción muy simple: usar siempre el promedio histórico. Un R² más alto
indica que el modelo explica mejor la variación de los conteos, aunque para Poisson
conviene revisarlo junto con MAE y desviación Poisson.

## 9. Graficar defectos reales contra predichos

Esta gráfica compara lo que realmente ocurrió contra lo que predijo el modelo.

- Cada punto es un lote de prueba.
- Eje horizontal: defectos reales.
- Eje vertical: defectos predichos.
- La línea diagonal representa predicción perfecta.

Si los puntos están cerca de la línea, el modelo predice bien. Si se alejan mucho,
hay errores grandes.

In [ ]:
comparacion = X_test.copy()
comparacion["defectos_reales"] = y_test
comparacion["defectos_predichos"] = pred_poisson
comparacion["error"] = comparacion["defectos_predichos"] - comparacion["defectos_reales"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=comparacion, x="defectos_reales", y="defectos_predichos",
                hue="turno", palette={"Día": "#167D9A", "Noche": "#DA9C20"},
                alpha=0.80, ax=ax)
limite = max(comparacion["defectos_reales"].max(), comparacion["defectos_predichos"].max()) + 1
ax.plot([0, limite], [0, limite], color="black", linestyle="--", linewidth=1)
ax.set_xlim(-0.2, limite)
ax.set_ylim(-0.2, limite)
ax.set_title("Defectos reales vs. defectos predichos")
ax.set_xlabel("Defectos reales")
ax.set_ylabel("Defectos predichos")
guardar_figura("03_reales_vs_predichos")

**Interpretación sencilla:** la línea punteada es el ideal. El modelo no necesita caer
exactamente sobre la línea para ser útil, pero sí queremos que esté razonablemente cerca.

## 10. Revisar errores del modelo

Aquí revisamos los errores de otra manera.

El error se calcula así:

**error = predicción - valor real**

- Si el error es positivo, el modelo predijo demasiados defectos.
- Si el error es negativo, el modelo predijo pocos defectos.
- Si el error está cerca de cero, la predicción fue cercana.

Esto ayuda a saber si el modelo tiende a sobreestimar o subestimar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(comparacion["error"], kde=True, color="#167D9A", ax=axes[0])
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_title("Distribución del error")
axes[0].set_xlabel("Error: predicho - real")

muestra = comparacion.sort_index().head(30).copy()
muestra["lote"] = range(1, len(muestra) + 1)
axes[1].plot(muestra["lote"], muestra["defectos_reales"], marker="o", label="Real")
axes[1].plot(muestra["lote"], muestra["defectos_predichos"], marker="o", label="Predicho")
axes[1].set_title("Primeros 30 lotes de prueba")
axes[1].set_xlabel("Lote de prueba")
axes[1].set_ylabel("Defectos")
axes[1].legend()

fig.tight_layout()
guardar_figura("04_errores_modelo")

## 11. Interpretar efectos de las variables

En regresión Poisson, los coeficientes se interpretan de forma multiplicativa.

Como estandarizamos las variables numéricas, el efecto se lee así:

**si una variable sube una desviación estándar, el número esperado de defectos se
multiplica por cierto valor.**

Ejemplo:

- multiplicador 1.20: el conteo esperado sube aproximadamente 20%.
- multiplicador 0.90: el conteo esperado baja aproximadamente 10%.
- multiplicador 1.00: casi no cambia.

Esta interpretación es asociación estadística, no causalidad.

In [ ]:
poisson = modelo_poisson.named_steps["poisson"]
nombres_categoricos = modelo_poisson.named_steps["preparacion"].named_transformers_["categoricas"].get_feature_names_out(variables_categoricas)
nombres_modelo = variables_numericas + list(nombres_categoricos)

coeficientes = pd.DataFrame({
    "variable": nombres_modelo,
    "coeficiente": poisson.coef_,
    "multiplicador": np.exp(poisson.coef_),
}).sort_values("multiplicador", ascending=False)

display(coeficientes.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
colores = np.where(coeficientes["multiplicador"] >= 1, "#C24E56", "#167D9A")
ax.barh(coeficientes["variable"], coeficientes["multiplicador"], color=colores)
ax.axvline(1, color="black", linewidth=1)
ax.set_xlabel("Multiplicador del número esperado de defectos")
ax.set_title("Efecto estimado por variable")
ax.invert_yaxis()
guardar_figura("05_coeficientes_poisson")

**Interpretación sencilla:** valores mayores que 1 se asocian con más defectos esperados.
Valores menores que 1 se asocian con menos defectos esperados. Esto no significa que
mover una variable cambie automáticamente los defectos, porque puede haber causas del
proceso que el modelo no conoce.

## 12. Predecir lotes nuevos

Ahora probamos el modelo con cinco lotes nuevos inventados.

La salida será el número esperado de defectos por lote. Como es una predicción promedio,
puede tener decimales. Por ejemplo, 2.6 significa que el modelo espera alrededor de
2 o 3 defectos en un lote con esas condiciones.

En manufactura, esto serviría para priorizar revisión, no para asegurar que el lote
tendrá exactamente ese número de defectos.

In [ ]:
lotes_nuevos = pd.DataFrame([
    [520, 64, 2.0, 115, "Día"],
    [580, 67, 2.5, 122, "Día"],
    [610, 70, 3.0, 128, "Noche"],
    [690, 72, 3.4, 132, "Noche"],
    [500, 66, 2.8, 118, "Noche"],
], columns=["tamaño_lote", "temperatura_c", "vibracion_mm_s", "velocidad_piezas_hora", "turno"],
   index=["Nuevo 1", "Nuevo 2", "Nuevo 3", "Nuevo 4", "Nuevo 5"])

lotes_nuevos["defectos_esperados"] = modelo_poisson.predict(lotes_nuevos)
display(lotes_nuevos.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.bar(lotes_nuevos.index, lotes_nuevos["defectos_esperados"], color="#C24E56")
ax.bar_label(barras, labels=[f"{v:.1f}" for v in lotes_nuevos["defectos_esperados"]], padding=4)
ax.set_title("Defectos esperados en lotes nuevos")
ax.set_ylabel("Defectos esperados")
guardar_figura("06_lotes_nuevos")

## 13. Generar conclusiones y guardar entregables

Este bloque genera conclusiones con los resultados reales de esta ejecución.

También guarda:

- datos simulados,
- predicciones de prueba,
- métricas,
- coeficientes,
- conclusiones,
- modelo entrenado,
- y un archivo ZIP.

Así el ejercicio queda listo para revisar, compartir o volver a ejecutar.

In [ ]:
mejora_mae = mae_simple - mae_poisson
mejora_rmse = rmse_simple - rmse_poisson
mejor_variable = coeficientes.iloc[0]["variable"]
mejor_multiplicador = coeficientes.iloc[0]["multiplicador"]
error_promedio = comparacion["error"].mean()

conclusiones = f"""
## Conclusiones

1. La regresión Poisson se usó porque la variable objetivo es un conteo:
   **defectos por lote**.

2. El modelo tuvo un error promedio absoluto de **{mae_poisson:.2f} defectos por lote**.
   La regla simple que siempre predice el promedio histórico tuvo **{mae_simple:.2f}**.
   Poisson mejoró el error promedio en **{mejora_mae:.2f} defectos por lote**.

3. La desviación Poisson del modelo fue **{dev_poisson:.3f}**, frente a
   **{dev_simple:.3f}** de la regla simple. En esta métrica, menor es mejor.

4. El RMSE del modelo fue **{rmse_poisson:.2f}**, frente a **{rmse_simple:.2f}**
   de la regla simple. La mejora fue de **{mejora_rmse:.2f} defectos** en esta métrica.
   RMSE ayuda a detectar si algunos lotes tienen errores grandes.

5. El R² del modelo fue **{r2_poisson:.3f}**. En lenguaje sencillo, indica qué tanto
   mejora el modelo frente a usar únicamente el promedio histórico. En conteos, R²
   debe verse como una métrica de apoyo, no como la única evidencia del desempeño.

6. El error promedio con signo fue **{error_promedio:.2f}**. Si es cercano a cero,
   el modelo no muestra una tendencia fuerte a sobreestimar o subestimar en conjunto.

7. La variable con mayor multiplicador estimado fue **{mejor_variable}**,
   con un multiplicador de **{mejor_multiplicador:.2f}**. Esto indica asociación
   con más defectos esperados, no causalidad.

8. La técnica sirve para estimar conteos esperados y priorizar revisión. No debe usarse
   para liberar producto automáticamente sin validación con datos reales, criterios de
   calidad y seguimiento del proceso.
"""

display(Markdown(conclusiones))

predicciones = comparacion.copy()
predicciones.to_csv(CARPETA / "predicciones_prueba.csv", index=False, encoding="utf-8-sig")
datos.to_csv(CARPETA / "datos_sinteticos_lotes.csv", index=False, encoding="utf-8-sig")
tabla_metricas.to_csv(CARPETA / "metricas.csv", index=False, encoding="utf-8-sig")
coeficientes.to_csv(CARPETA / "coeficientes.csv", index=False, encoding="utf-8-sig")
lotes_nuevos.to_csv(CARPETA / "lotes_nuevos.csv", encoding="utf-8-sig")
(CARPETA / "conclusiones.md").write_text(conclusiones, encoding="utf-8")
joblib.dump(modelo_poisson, CARPETA / "modelo_poisson.joblib")

metadatos = {
    "tipo": "Ejercicio didáctico con datos sintéticos",
    "fecha": "2026-09-06",
    "filas": len(datos),
    "objetivo": "defectos_lote",
    "variables": list(X.columns),
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
}
(CARPETA / "metadatos.json").write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding="utf-8")

ruta_zip = shutil.make_archive(str(CARPETA), "zip", root_dir=CARPETA)
print("Resultados guardados en:", CARPETA.resolve())
print("ZIP generado:", ruta_zip)

## Función Poisson explicada de forma sencilla

La función del modelo Poisson sirve para convertir las variables del proceso en un
número esperado de defectos. No entrega una categoría; entrega una cantidad esperada.

La forma general es:

```text
log(lambda)=b0+b1x1+b2x2+b3x3+...
```

En notación matemática se escribe así:

$$
\\log(\\lambda) =
b_0 + b_1x_1 + b_2x_2 + b_3x_3 + ...
$$

Donde:

- \(\\lambda\) se lee "lambda" y significa **número esperado de defectos**.
- \(\\log(\\lambda)\) es el logaritmo del número esperado de defectos.
- \(b_0\) es el punto de partida del modelo.
- \(b_1, b_2, b_3\) son pesos aprendidos a partir de los datos históricos.
- \(x_1, x_2, x_3\) son variables del proceso, como temperatura, vibración o velocidad.

La parte izquierda usa logaritmo por una razón práctica: el modelo necesita asegurar
que el resultado final sea positivo. No tendría sentido predecir -2 defectos. Por eso
primero trabaja en escala logarítmica y luego regresa a la escala natural del conteo.

Como el modelo usa logaritmo, después se aplica exponencial:

$$
\\lambda = e^{b_0 + b_1x_1 + b_2x_2 + b_3x_3 + ...}
$$

Lectura sencilla:

**el modelo suma efectos de las variables, los transforma con exponencial y obtiene
un número esperado de defectos.**

Si el resultado final es 2.8, se interpreta como:

**para un lote con esas condiciones, el modelo espera alrededor de 2.8 defectos.**

Eso no significa que el lote tendrá exactamente 2.8 defectos. Significa que, según el
patrón aprendido en el historial, ese es el conteo esperado.

### Cómo leer los coeficientes

En un modelo Poisson, los coeficientes se leen mejor con su exponencial:

$$
multiplicador = e^{coeficiente}
$$

Ejemplos sencillos:

- Si el multiplicador es 1.20, el modelo espera aproximadamente 20% más defectos.
- Si el multiplicador es 0.90, el modelo espera aproximadamente 10% menos defectos.
- Si el multiplicador es 1.00, el efecto estimado es casi neutro.

En este notebook las variables numéricas fueron estandarizadas antes de entrenar.
Por eso, cuando leemos un multiplicador de temperatura, vibración o velocidad, lo
interpretamos como el cambio esperado cuando esa variable sube aproximadamente una
desviación estándar, manteniendo las demás variables constantes.

### Qué significa "manteniendo las demás variables constantes"

Significa que el modelo intenta aislar el efecto estadístico de una variable mientras
las otras no cambian. Por ejemplo, pregunta:

**qué pasa con los defectos esperados si sube la vibración, pero el tamaño de lote,
la temperatura, la velocidad y el turno se mantienen iguales?**

Esta lectura ayuda a interpretar el modelo, pero no demuestra causalidad. Si vibración
aparece asociada con más defectos, eso no prueba por sí solo que bajar la vibración
reducirá defectos. Para afirmar causalidad se necesitaría conocimiento del proceso,
experimentación o un diseño de análisis más fuerte.

### Qué debe recordar un principiante

1. La regresión Poisson predice conteos esperados.
2. El resultado nunca debe ser negativo.
3. La función usa logaritmo y exponencial para mantener la predicción en valores positivos.
4. Los coeficientes se interpretan mejor como multiplicadores.
5. Las métricas ayudan a saber qué tan cerca están los conteos predichos de los reales.
6. El modelo apoya decisiones, pero necesita validación antes de usarse en operación.

## Cómo adaptar este ejercicio a datos reales

Para usarlo con datos reales, reemplace el bloque 3 por:

```python
datos = pd.read_csv("mis_lotes.csv")
```

La tabla real debe tener una fila por lote y una columna con el número de defectos.
También debe tener variables disponibles antes de tomar la decisión.

Recomendaciones:

1. Use datos de varios días, turnos, máquinas o lotes.
2. Revise unidades y valores faltantes.
3. Separe la prueba por fecha si quiere medir desempeño futuro.
4. Si los lotes tienen tamaños muy diferentes, revise tasas o modelos con exposición.
5. Si hay demasiados ceros, explore modelos especiales para exceso de ceros.
6. Si la variación es muy alta, compare contra alternativas como regresión binomial negativa.

## Preguntas para clase

- ¿Por qué Poisson se usa para conteos?
- ¿Qué significa un error promedio de 1.5 defectos por lote?
- ¿Por qué una predicción puede tener decimales si los defectos reales son enteros?
- ¿Qué métrica sería más fácil de explicar a un supervisor de producción: MAE, RMSE, R² o desviación Poisson?

## Fuentes para ampliar

- Documentación oficial de `PoissonRegressor` en scikit-learn:
  https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.PoissonRegressor.html.
- Documentación oficial de modelos lineales generalizados en scikit-learn:
  https://scikit-learn.org/stable/modules/linear_model.html#generalized-linear-models.

Los datos de este notebook son sintéticos y tienen fines didácticos.